# 06 · 多重检验与 Bootstrap Multiple Testing & Bootstrap

> **能力标签**：SH6（概率与统计 / Probability & statistics）

## 目标 Objectives

完成本课后，你应该能够：

1. 理解**多重检验问题**（multiple testing problem）：为什么同时做多个检验会增加假阳性率。
2. 实现并应用 **Bonferroni 校正**（控制 FWER）。
3. 实现并应用 **Benjamini–Hochberg FDR 校正**（控制 FDR）。
4. 用 **Bootstrap** 方法计算均值的置信区间（无需正态假设）。

> 对应能力：**SH6**


## 概念讲解 Concepts

### 多重检验问题 Multiple Testing Problem

当同时进行 $m$ 个检验时，即使每次检验的显著性水平为 $\alpha$，**族错误率**（Family-Wise Error Rate, FWER）可急剧升高：

$$\text{FWER} = 1 - (1-\alpha)^m \approx m\alpha$$（当 $m\alpha$ 很小时）

例如：$m=100, \alpha=0.05$ → 期望约 5 个假阳性！

---

### Bonferroni 校正

最简单的 FWER 控制方法：将显著性阈值降低 $m$ 倍：

$$p_i < \frac{\alpha}{m} \Rightarrow \text{拒绝} H_0^{(i)}$$

- 优点：严格控制 FWER，计算简单
- 缺点：保守（功效低），适合少量检验

---

### Benjamini–Hochberg FDR 校正

**假发现率**（False Discovery Rate, FDR）= 被拒绝中的假阳性比例期望。

BH 步骤（设 $p_{(1)} \leq \ldots \leq p_{(m)}$）：

1. 对 p 值排序：$p_{(1)} \leq p_{(2)} \leq \cdots \leq p_{(m)}$
2. 找最大 $k$ 使得 $p_{(k)} \leq \frac{k}{m} \alpha$
3. 拒绝所有 $i \leq k$ 对应的原假设

BH 校正控制 FDR ≤ α（在独立检验假设下）。相比 Bonferroni，功效更高。

---

### Bootstrap 置信区间

Bootstrap 是一种**非参数重采样**方法，无需分布假设：

1. 从原始数据（$n$ 个）**有放回抽样** $B$ 次，每次 $n$ 个
2. 对每个 bootstrap 样本计算统计量（如均值）
3. 用 bootstrap 样本统计量的分位数构建置信区间

95% Bootstrap CI：$[Q_{2.5\%}(\hat{\theta}^*), Q_{97.5\%}(\hat{\theta}^*)]$


## 示例 Worked Example

In [ ]:
import numpy as np

def bonferroni(pvals, alpha: float = 0.05):
    """Bonferroni 校正：返回布尔数组（是否拒绝原假设）。"""
    p = np.asarray(pvals, dtype=float)
    return p < (alpha / len(p))


def bh_fdr(pvals, alpha: float = 0.05):
    """Benjamini–Hochberg FDR：返回布尔数组（保持原顺序）。"""
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    thresh = alpha * (np.arange(1, n + 1) / n)
    below = ranked <= thresh
    reject = np.zeros(n, dtype=bool)
    if below.any():
        kmax = int(np.max(np.where(below)[0]))
        reject[order[: kmax + 1]] = True
    return reject


# 模拟场景：100 个基因关联检验，10 个真实信号
rng = np.random.default_rng(42)
m = 100
n_true = 10    # 真实有效的检验
n_null = m - n_true

# 构造 p 值：真实信号（小 p）+ 零假设（均匀分布 p）
from scipy import stats
pvals_signal = rng.uniform(0, 0.01, n_true)   # 真实信号，小 p 值
pvals_null   = rng.uniform(0, 1,    n_null)    # 零假设，均匀分布
pvals = np.concatenate([pvals_signal, pvals_null])

# 校正
reject_none = pvals < 0.05
reject_bonf = bonferroni(pvals, alpha=0.05)
reject_bh   = bh_fdr(pvals, alpha=0.05)

# 真实标签
is_signal = np.zeros(m, dtype=bool)
is_signal[:n_true] = True

def count_errors(reject, is_signal):
    tp = (reject & is_signal).sum()
    fp = (reject & ~is_signal).sum()
    fn = (~reject & is_signal).sum()
    return tp, fp, fn

tp0, fp0, fn0 = count_errors(reject_none, is_signal)
tp_b, fp_b, fn_b = count_errors(reject_bonf, is_signal)
tp_h, fp_h, fn_h = count_errors(reject_bh, is_signal)

print("=== 多重检验校正比较 ===")
print(f"{'方法':15} {'拒绝数':>6} {'TP':>4} {'FP':>4} {'FN':>4}")
print("-" * 40)
print(f"{'无校正':15} {reject_none.sum():>6} {tp0:>4} {fp0:>4} {fn0:>4}")
print(f"{'Bonferroni':15} {reject_bonf.sum():>6} {tp_b:>4} {fp_b:>4} {fn_b:>4}")
print(f"{'BH-FDR':15} {reject_bh.sum():>6} {tp_h:>4} {fp_h:>4} {fn_h:>4}")
print(f"\n结论：BH-FDR 在控制假阳性的同时，比 Bonferroni 发现更多真实信号")


In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tempfile
from pathlib import Path

# Bootstrap 置信区间
rng = np.random.default_rng(0)
n = 80
# 非正态数据：指数分布
data = rng.exponential(scale=3.0, size=n)

# Bootstrap
B = 5000
boot_idx = rng.integers(0, n, size=(B, n))
boot_means = data[boot_idx].mean(axis=1)

ci_low  = float(np.percentile(boot_means, 2.5))
ci_high = float(np.percentile(boot_means, 97.5))
sample_mean = data.mean()

print("=== Bootstrap 均值置信区间 (95%) ===")
print(f"  样本均值 = {sample_mean:.4f}")
print(f"  Bootstrap CI: [{ci_low:.4f}, {ci_high:.4f}]")
print(f"  区间宽度 = {ci_high - ci_low:.4f}")
print(f"  真实 μ = 3.0 → {'包含在 CI 内' if ci_low <= 3.0 <= ci_high else '未包含（抽样误差）'}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 原始数据分布
axes[0].hist(data, bins=20, density=True, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].set_title("原始数据（指数分布，非正态）")
axes[0].set_xlabel("值")
axes[0].set_ylabel("密度")

# Bootstrap 均值分布
axes[1].hist(boot_means, bins=40, density=True, color="darkorange", edgecolor="white", alpha=0.8)
axes[1].axvline(ci_low,       color="red",   ls="--", lw=1.5, label=f"2.5% = {ci_low:.3f}")
axes[1].axvline(ci_high,      color="red",   ls="--", lw=1.5, label=f"97.5% = {ci_high:.3f}")
axes[1].axvline(sample_mean,  color="blue",  ls="-",  lw=2,   label=f"样本均值 = {sample_mean:.3f}")
axes[1].set_title("Bootstrap 均值分布（CLT 近似正态）")
axes[1].set_xlabel("Bootstrap 均值")
axes[1].set_ylabel("密度")
axes[1].legend(fontsize=8)

fig.tight_layout()
_tmpdir = tempfile.mkdtemp()
_outpath = Path(_tmpdir) / "bootstrap_ci_demo.png"
fig.savefig(_outpath, dpi=80)
plt.close(fig)
print("\n图像已保存到:", _outpath)


## 动手 Exercise

In [ ]:
import numpy as np

def bonferroni(pvals, alpha: float = 0.05):
    p = np.asarray(pvals, dtype=float)
    return p < (alpha / len(p))

def bh_fdr(pvals, alpha: float = 0.05):
    p = np.asarray(pvals, dtype=float)
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    thresh = alpha * (np.arange(1, n + 1) / n)
    below = ranked <= thresh
    reject = np.zeros(n, dtype=bool)
    if below.any():
        kmax = int(np.max(np.where(below)[0]))
        reject[order[: kmax + 1]] = True
    return reject

# 练习：GWAS 场景 — 基因组关联研究（简化版）
rng = np.random.default_rng(33)
m = 200   # SNP 数量
pvals = rng.uniform(0, 1, m)
# 注入 5 个真实信号
pvals[:5] = rng.uniform(1e-6, 1e-4, 5)
np.random.default_rng(33).shuffle(pvals)

rej_bonf = bonferroni(pvals, alpha=0.05)
rej_bh   = bh_fdr(pvals, alpha=0.05)

print(f"共 {m} 个 SNP，其中注入 5 个信号")
print(f"Bonferroni 拒绝: {rej_bonf.sum()} 个")
print(f"BH-FDR    拒绝: {rej_bh.sum()} 个")
print(f"\nBonferroni 阈值: {0.05/m:.2e}")
print(f"拒绝的最大 p 值 (BH): {pvals[rej_bh].max():.4e}" if rej_bh.any() else "BH: 无拒绝")

# Bootstrap CI 练习
data = rng.exponential(2.0, 50)
B = 2000
boot_means = np.array([rng.choice(data, size=50, replace=True).mean() for _ in range(B)])
ci = np.percentile(boot_means, [2.5, 97.5])
print(f"\nBootstrap 95% CI (n=50, B={B}): [{ci[0]:.4f}, {ci[1]:.4f}]")
print(f"样本均值: {data.mean():.4f}, 真实 μ=2.0")


## 延伸阅读 Further Reading

1. **Benjamini & Hochberg (1995)**：FDR 原始论文，JRSS-B 57(1):289–300
2. **《All of Statistics》— Wasserman**：第 11 章（多重检验）
3. **StatQuest: FDR and Benjamini-Hochberg**（YouTube）
4. **Bootstrap 方法**：Efron & Tibshirani 《An Introduction to the Bootstrap》（经典教材）
5. **GWAS 多重检验**：Bonferroni 在全基因组检验中的应用（~5×10⁻⁸ 阈值的由来）


## 关联练习 Related Assignments

本课对应练习包：**`w4-multiple-testing`**（实现 `bonferroni()` 与 `bh_fdr()` 函数）

```bash
python check.py w4-multiple-testing
```

> 能力标签：**SH6** · threshold ≥ 0.7
